# Fine-Tuning Qwen2.5-1.5B-Instruct Using AraFinNews Dataset

## Problem Definition

Fine-tune an affordable open-weight LLM (Qwen2.5-1.5B-Instruct) to translate Arabic financial news into English while extracting structured financial intelligence in a constrained JSON schema.

## Dataset Filtering, Synthetic Supervision, Validation

<img src="https://drive.google.com/uc?id=1WAxAD7baWPBlPiyVu_1s7BBUiypdagPL" alt="">

**_NOTE:_**  Although using two teachers, one for extraction and the other one for translation, may results in better labeling, I will use one teacher here as the first trial.


## Fine-Tuning Process

<img src="https://drive.google.com/uc?id=1W3QP2z1W5Y1Og_aQ3wENCSXZCB4OFA50" alt="">

# Install Dependencies

In [ ]:
# This command solve a conflict between torch and torchaudio versions
!pip install -q \
    torch==2.7.1 \
    torchvision==0.22.1 \
    torchaudio==2.7.1 \
    --index-url https://download.pytorch.org/whl/cu128

!pip install -q \
    bitsandbytes==0.50.2 \
    transformers==5.17.0 \
    datasets==5.0.1 \
    optimum==2.3.0 \
    openai==3.13.0 \
    wandb==0.30.0 \
    json-repair==0.63.4 \
    vllm==0.29.0

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

# Imports

In [ ]:
from dotenv import load_dotenv
import wandb
from huggingface_hub import login
import json
import os
from os.path import join
from pathlib import Path
import random
from tqdm.auto import tqdm
import requests
import time

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime


from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch


# Setup HF and WANDB

In [ ]:
PROJECT_NAME = "Fine-Tuning-Qwen2.5-1.5B-Instruct-AraFinNews"
DATASET_NAME = "drelhaj/AraFinNews"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda"

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
WANDB = secrets.get_secret('WANDB_API_KEY')
SOVEREIGNEG = secrets.get_secret('SOVEREIGNEG_API_KEY')

In [ ]:
# Log in to HuggingFace

login(HF_TOKEN)

In [ ]:
# Log in to Weights & Biases
os.environ["WANDB_API_KEY"] = WANDB
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

# Loading AraFinNews Dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['validation']
test = dataset['test']

In [ ]:
input1 = train[1]["article"]
input1

# Structured Output

In [ ]:
class FinancialMetric(BaseModel):
    metric: str = Field(description="The financial metric being reported, such as revenue, net profit, operating income, or earnings per share.")
    value: str = Field(description="The reported value of the financial metric, preserving the value as stated in the article.")
    currency: str | None = Field(
        default=None,
        description="The currency associated with the financial metric, such as USD, EUR, or SAR. Null if no currency is explicitly stated."
    )
    period: str | None = Field(
        default=None,
        description="The financial reporting period associated with the metric, such as Q1 2025, Q2 2025, FY 2024, or 2025. Null if no period is explicitly stated."
    )


class FinancialEvent(BaseModel):
    event_type: str = Field(description="The type of financial event described in the article, such as profit_increase, profit_decrease, revenue_growth, acquisition, merger, investment, or dividend.")
    company: str | None = Field(
        default=None,
        description="The company directly associated with the financial event. Null if no specific company can be identified."
    )
    percentage: str | None = Field(
        default=None,
        description="The percentage change associated with the event, such as 15%, 20%, or -7%. Null if the event does not include a percentage."
    )


class FinancialIntelligenceResponse(BaseModel):
    title_ar: str = Field(
        description="The original Arabic title of the financial news article."
    )
    title_en: str = Field(
        description="An accurate English translation of the Arabic article title."
    )
    translation: str = Field(
        description="A faithful English translation of the complete Arabic financial news article."
    )
    companies: List[str] = Field(
        description="Names of companies or other business organizations explicitly mentioned in the article."
    )
    people: List[str] = Field(
        description="Names of people explicitly mentioned in the article."
    )
    countries: List[str] = Field(
        description="Names of countries explicitly mentioned in the article."
    )

    locations: List[str] = Field(
        description="Cities, regions, or other geographic locations explicitly mentioned in the article."
    )
    financial_events: List[FinancialEvent] = Field(
        description="Financial events explicitly described in the article, including events such as profit changes, revenue growth, acquisitions, mergers, investments, or dividends."
    )
    financial_metrics: List[FinancialMetric] = Field(
        description="Financial metrics explicitly reported in the article, including their values, currencies, and reporting periods when available."
    )
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The overall financial sentiment of the article: positive, negative, or neutral."
    )

# Test Example

In [ ]:

messages = [
    {
        "role": "system",
        "content": "\n".join([
            "Your an expert in translation and extraction of financial events and metrics.",
            "You will be provided by an Arabic text associated with a Pydantic scheme.",
            "Generate the ouptut in the same input text language.",
            "You have to extract JSON details from text according the Pydantic details.",
            "Extract details as mentioned in text.",
            "Do not generate any introduction or conclusion."
        ])
    },
    {
        "role": "user",
        "content": "\n".join([
            "## Input Text:",
            input1.strip(),
            "",

            "## Pydantic Details:",
            json.dumps(
                FinancialIntelligenceResponse.model_json_schema(), 
                ensure_ascii=False
            ),
            "",

            "## Story Details:",
            "```json"
        ])
    }
]

# Load Base Model

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
  BASE_MODEL,
  device_map='auto',
  dtype=None 
)

In [ ]:
base_model

# Evaluate Base Model

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(DEVICE)

generated_ids = base_model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

# Evaluate Teacher Model

In [ ]:
from openai import OpenAI

TEACHER_MODEL = "gpt-4o-mini"

client = OpenAI(
    base_url="https://backend.sovereigneg.com/v1",
    api_key=SOVEREIGNEG,
)

response = client.chat.completions.parse(
    model=TEACHER_MODEL,
    messages=messages,
    response_format=FinancialIntelligenceResponse,
)

result = response.choices[0].message.parsed
print(result.model_dump_json(indent=2))

# Synthetic Supervision / Knowledge Distillation

In [ ]:
SYSTEM_PROMPT = """
You are a professional Arabic financial intelligence system.

Analyze Arabic financial news articles.

Your tasks are:


1. Translate the article title into English.
2. Translate the complete article into accurate English.
Generate the ouptut of the following in the same input text language.
3. Extract companies.
4. Extract people.
5. Extract countries.
6. Extract geographic locations.
7. Extract explicit financial events.
8. Extract explicit financial metrics.
9. Determine the overall financial sentiment.

IMPORTANT RULES:

- Extract ONLY information explicitly stated in the article.
- Do NOT infer facts that are not present.
- Do NOT hallucinate entities, values, currencies, dates, or events.
- Preserve financial numbers exactly as stated whenever possible.
- If a category has no entities, return an empty list.
- If an optional attribute is not explicitly stated, return null.
- Use the predefined event types.
- The translation must preserve the meaning of the original Arabic article.
"""

In [ ]:
TRAIN_SAMPLES = 2500
VAL_SAMPLES = 50
TEST_SAMPLES = 50

SEED = 42

train_clean = train.filter(lambda x: x["article"] is not None and x["title"] is not None)
val_clean = val.filter(lambda x: x["article"] is not None and x["title"] is not None)
test_clean = test.filter(lambda x: x["article"] is not None and x["title"] is not None)

train_sample = train_clean.shuffle(seed=SEED).select(
    range(min(TRAIN_SAMPLES, len(train_clean)))
)

val_sample = val_clean.shuffle(seed=SEED).select(
    range(min(VAL_SAMPLES, len(val_clean)))
)

test_sample = test_clean.shuffle(seed=SEED).select(
    range(min(TEST_SAMPLES, len(test_clean)))
)

print(train_sample)
print(val_sample)
print(test_sample)

In [ ]:
def generate_label(input: dict):
  prompt = f"""
Analyze the following Arabic financial news article.

TITLE:
{input["title"]}

ARTICLE:
{input["article"]}
"""
  response = client.chat.completions.parse(
        model=TEACHER_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        response_format=FinancialIntelligenceResponse,
    )

  return response

def append_jsonl(path: str, record: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(
            record,
            ensure_ascii=False
        ) + "\n")

def load_processed_ids(path: str) -> set:

    processed_ids = set()

    if not Path(path).exists():
        return processed_ids

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            processed_ids.add(str(record["id"]))

    return processed_ids


In [ ]:
def generate_dataset(dataset, output_path: str,
    sleep_seconds: float = 0.2):
    price_per_1m_input_tokens = 8.47 # EGP
    price_per_1m_output_tokens = 33.89 # EGP

    total_prompt_tokens = 0
    total_completion_tokens = 0
    ix = 0

    Path(output_path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    processed_ids = load_processed_ids(output_path)

    total = len(dataset)

    for i, article in tqdm(enumerate(dataset), total=total):
        
        article_id = str(article["id"])

        # Resume support
        if article_id in processed_ids:
            continue

        try:
            curr_response = generate_label(article)
            # print(curr_response.usage)
            # print(curr_response.usage.prompt_tokens)
            # print(curr_response.usage.completion_tokens)
            # break
            result = curr_response.choices[0].message.parsed
            

            record = {
                "id": article["id"],
                "title_ar": article["title"],
                "date": article["date"],
                "url": article["url"],
                "article": article["article"],
                **result.model_dump(),
            }

            append_jsonl(
                output_path,
                record,
            )

            total_prompt_tokens += int(curr_response.usage.prompt_tokens)
            total_completion_tokens += int(curr_response.usage.completion_tokens)
            
            time.sleep(sleep_seconds)

        except Exception as e:

            print(
                f"[{i + 1}/{total}] "
                f"FAILED: {article_id} | {e}"
            )

            continue

        ix += 1
        if(ix % 50) == 0:
          cost_input = (total_prompt_tokens / 1_000_000) * price_per_1m_input_tokens
          cost_output = (total_completion_tokens / 1_000_000) * price_per_1m_output_tokens
          total_cost = cost_input + cost_output

          print(f"Iteration {ix}: Total Cost = {total_cost:.4f}EGP ")

In [ ]:
generate_dataset(
    train_sample,
    "synthetic/train.jsonl",
)

In [ ]:

generate_dataset(
    val_sample,
    "synthetic/val.jsonl",
)

In [ ]:
generate_dataset(
    test_sample,
    "synthetic/test.jsonl",
)

# LLaMA-Factory Data Formating

In [ ]:
FINE_TUNING_SYS_PROMPT = "\n".join([
    "You are a professional NLP data parser.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."
])

In [ ]:
TASK = "\n".join([
    "Your task is to translate and extract financial events and metrics.",
    "You will be provided by an Arabic article associated with an Output Scheme.",
    "Generate the ouptut in the same input text language.",
    "You have to extract JSON details from text according the Output Scheme details.",
    "Extract details as mentioned in text."
])

In [ ]:
def llamafactory_sft_data(
    data_path: str, 
    output_dir: str, 
    file_name: str,
    shuffle: bool = True,
    seed: int = SEED
):
  llamafactory_finetuning_data = []
  for line in open(data_path):
    if line.strip() == "":
      continue
    
    record = json.loads(line.strip())

    llamafactory_finetuning_data.append({
        "system": FINE_TUNING_SYS_PROMPT,
        "instruction": "\n".join([
            f"# Article Title: {record["title_ar"]}",
            "# Article: ",
            record["article"],

            "# Task:",
            TASK,

            "# Output Scheme:",
            json.dumps(
                FinancialIntelligenceResponse.model_json_schema(), 
                ensure_ascii=False
            ),
            "",

            "# Output JSON:",
            "```json"

        ]),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps({
               "title_ar": record["title_ar"],
               "title_en": record["title_en"],
               "translation": record["translation"],
               "companies": record["companies"],
               "people": record["people"],
               "countries": record["countries"],
               "locations": record["locations"],
               "financial_events": record["financial_events"],
               "financial_metrics": record["financial_metrics"], 
               "sentiment": record["sentiment"]
            }, ensure_ascii=True, default=str),
            "```"
        ]),
        "history": []
    })

  if shuffle:
    random.Random(seed).shuffle(llamafactory_finetuning_data)

  os.makedirs(output_dir,  exist_ok=True)

  with open(join(output_dir, file_name), "w", encoding="utf-8") as dest:
    json.dump(llamafactory_finetuning_data, dest, ensure_ascii=False, default=str)
  
  print(f"Len of processed data: {len(llamafactory_finetuning_data)}")
  print(f"Saved to: {output_dir}/{file_name}")

In [ ]:
llamafactory_sft_data(
    data_path="/kaggle/working/synthetic/train.jsonl", 
    output_dir="/kaggle/working/sft_data", 
    file_name="train.json"
)

In [ ]:
llamafactory_sft_data(
    data_path="/kaggle/working/synthetic/val.jsonl", 
    output_dir="/kaggle/working/sft_data", 
    file_name="val.json"
)

# Fine-Tuning using LLaMA-Factory

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

In [ ]:
# configure Llama-Factory with our datasets
# update /LLaMA-Factory/data/dataset_info.json 
# Append the following
# ```
   "news_finetune_train": {
        "file_name": "/gdrive/MyDrive/youtube-resources/llm-finetuning/datasets/llamafactory-finetune-data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "news_finetune_val": {
        "file_name": "/gdrive/MyDrive/youtube-resources/llm-finetuning/datasets/llamafactory-finetune-data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
# ```

In [ ]:
path = Path("/kaggle/working/LLaMA-Factory/data/dataset_info.json")


with path.open("r", encoding="utf-8") as f:
    dataset_info = json.load(f)

# Add your datasets
dataset_info.update({
    "news_finetune_train": {
        "file_name": "/kaggle/working/sft_data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "news_finetune_val": {
        "file_name": "/kaggle/working/sft_data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
})

with path.open("w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"Updated: {path}")

In [ ]:
with path.open("r", encoding="utf-8") as f:
    dataset_info = json.load(f)

print(json.dumps({
    "news_finetune_train": dataset_info["news_finetune_train"],
    "news_finetune_val": dataset_info["news_finetune_val"]
}, ensure_ascii=False, indent=2))

In [ ]:
%%writefile /kaggle/working/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 8

### output
# resume_from_checkpoint: 
output_dir: /kaggle/working/qwen25-1.5b-arafinnews-lora/
logging_steps: 5
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 20

report_to: wandb
run_name: news-finetune-llamafactory-2

push_to_hub: true
export_hub_model_id: "abdallahsalah0/qwen2.5-1.5b-arafinnews-lora"
hub_private_repo: false
hub_strategy: checkpoint

In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /kaggle/working/LLaMA-Factory/examples/train_lora/news_finetune.yaml

# Fine-Tuned Model Inference

In [ ]:
test = { 
    "title_ar": "ارتفاع صادرات إيطاليا إلى الصين 3 أضعاف في عام", 
    "article": "ارتفعت صادرات إيطاليا إلى الصين ثلاثة أضعاف في عام مع زيادة صادرات الأدوية بصورة حادة، وهو أمر يصعب تفسيره حتى من قبل الخبراء.  وحسبما ذكرت بلومبرج، ارتفعت الشحنات الإيطالية الخارجية الموجهة للصين 131% في فبراير على أساس سنوي إلى أكثر من 3 مليارات يورو (3.3 مليار دولار)، بعدما زادت 137% في الشهر السابق، إذ بلغت الصادرات مليار يورو في يناير 2022. وشكلت صادرات الأدوية ما يقرب من ثلثي إجمالي الصادرات الإيطالية إلى الصين، إذ ارتفعت صادراتها إلى 1.84 مليار يورو في فبراير من 98.5 مليون يورو في نفس الفترة من العام الماضي. ورغم أن إيطاليا هي الدولة الوحيدة بين أعضاء مجموعة السبع التي وقعت على مبادرة الحزام والطريق، إلا أن الفوائد الاقتصادية لهذا التحالف كانت محدودة منذ عام 2019. ويصعب تفسير ارتفاع الصادرات بعدما تعرضت العلاقات بين البلدين لفتور شديد في عهد رئيس الوزراء الإيطالي ماريو دراغي، أما رئيسة الوزراء الحالية جيورجيا ميلوني فقد أبلغت مسؤولين أمريكين بأنها ستنسحب من اتفاق الصين قبل نهاية العام الحالي.", 
}

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": FINE_TUNING_SYS_PROMPT
    },
    {
        "role": "user",
        "content": "\n".join([
            f"# Article Title: {test["title_ar"]}",
            "# Article: ",
            test["article"],

            "# Task:",
            TASK,

            "# Output Scheme:",
            json.dumps(
                FinancialIntelligenceResponse.model_json_schema(), 
                ensure_ascii=False,
                default=str
            )
        ])
    }
]

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "abdallahsalah0/qwen25-1.5b-arafinnews-lora"

DEVICE = "cuda"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype = None
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model.load_adapter(ADAPTER)

In [ ]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt",
    ).to(DEVICE)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1500,
        do_sample=False,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            model_inputs.input_ids,
            generated_ids
        )
    ]

    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return response

In [ ]:
response = generate_resp(messages)
json.loads(response[7:-4])